# Applio — экспериментальные вокодеры

Нажми **Среда выполнения → Выполнить всё**. Блокнот сам установит окружение, прогонит тестовые ячейки через `nbclient`, при необходимости синхронизирует Drive и запустит сервер в фоне.


In [ ]:
# @title 1. Установить Applio Vocoders и тестовый раннер
rebuild_environment = False  # @param {type:"boolean"}

from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/egor125552/Applio.git"
BRANCH = "exp/vocoders"
APP_DIR = Path("/content/Applio-vocoders")
VENV_DIR = Path("/content/applio-vocoders-env")
PYTHON = VENV_DIR / "bin" / "python"
MARKER = VENV_DIR / ".applio_commit"
REQ_FILE = Path("/content/applio-vocoders-requirements.txt")
KERNEL_NAME = "applio-vocoders"
ENV_SCHEMA = "run-all-v2"

if not Path("/content").exists():
    raise RuntimeError("Этот блокнот рассчитан на Google Colab.")

def run(command, cwd=None):
    command = [str(x) for x in command]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

has_gpu = shutil.which("nvidia-smi") is not None
if has_gpu:
    run(["nvidia-smi"])
else:
    print("⚠️ GPU не найден: приложение запустится, но будет медленным.")

run(["apt-get", "update", "-qq"])
run([
    "apt-get", "install", "-y", "-qq",
    "git", "git-lfs", "ffmpeg", "libsndfile1",
    "portaudio19-dev", "build-essential"
])

if (APP_DIR / ".git").exists():
    run(["git", "remote", "set-url", "origin", REPO_URL], APP_DIR)
    run(["git", "fetch", "--depth", "1", "origin", BRANCH], APP_DIR)
    run(["git", "checkout", "-B", BRANCH, "FETCH_HEAD"], APP_DIR)
    run(["git", "reset", "--hard", "FETCH_HEAD"], APP_DIR)
    run(["git", "clean", "-fdx"], APP_DIR)
else:
    if APP_DIR.exists():
        shutil.rmtree(APP_DIR)
    run([
        "git", "clone", "--depth", "1", "--branch", BRANCH,
        "--single-branch", REPO_URL, APP_DIR
    ])

run(["git", "lfs", "pull"], APP_DIR)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=APP_DIR, text=True
).strip()
print("Commit:", commit)

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"])
UV = [sys.executable, "-m", "uv"]
current = (
    PYTHON.exists()
    and MARKER.exists()
    and MARKER.read_text(encoding="utf-8").strip() == f"{commit}:{ENV_SCHEMA}"
)

if rebuild_environment or not current:
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)

    run(UV + ["python", "install", "3.10"])
    run(UV + ["venv", "--seed", "--python", "3.10", VENV_DIR])
    run([PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

    torch_index = (
        "https://download.pytorch.org/whl/cu121"
        if has_gpu
        else "https://download.pytorch.org/whl/cpu"
    )
    run([
        PYTHON, "-m", "pip", "install", "--no-cache-dir",
        "--index-url", torch_index,
        "torch==2.3.1", "torchvision==0.18.1", "torchaudio==2.3.1"
    ])

    skipped_names = {"torch", "torchvision", "torchaudio", "treon"}
    requirements = []
    for raw_line in (APP_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines():
        stripped = raw_line.strip()
        package_name = stripped.split(";", 1)[0].strip()
        package_name = package_name.split("==", 1)[0].split(">=", 1)[0].split("<=", 1)[0]
        package_name = package_name.split("~=", 1)[0].strip().lower()
        if package_name in skipped_names:
            continue
        requirements.append(raw_line)

    requirements += [
        "",
        "# Colab and notebook validation",
        "pydantic==2.8.2",
        "fastapi==0.112.0",
        "starlette==0.37.2",
        "nbformat>=5.10,<6",
        "nbclient>=0.10,<0.11",
        "jupyter-client>=8.6,<9",
        "ipykernel>=6.29,<7",
    ]
    REQ_FILE.write_text("\n".join(requirements) + "\n", encoding="utf-8")
    run([PYTHON, "-m", "pip", "install", "--no-cache-dir", "-r", REQ_FILE])
    run([PYTHON, "-m", "pip", "check"])
    MARKER.write_text(f"{commit}:{ENV_SCHEMA}\n", encoding="utf-8")
else:
    print("✅ Окружение уже установлено для этого commit.")

run([
    PYTHON, "-m", "ipykernel", "install", "--user",
    "--name", KERNEL_NAME,
    "--display-name", "Applio Vocoders Test"
])

print("✅ Установка и тестовый Jupyter-kernel готовы.")


In [ ]:
# @title 2. Фоновый прогон тестовых ячеек через nbclient
from pathlib import Path
import subprocess
import textwrap

APP_DIR = Path("/content/Applio-vocoders")
PYTHON = Path("/content/applio-vocoders-env/bin/python")
RUNNER = Path("/content/run_applio_vocoders_smoke.py")
REPORT = Path("/content/applio-vocoders-smoke.executed.ipynb")
KERNEL_NAME = "applio-vocoders"

if not PYTHON.exists():
    raise RuntimeError("Установочная ячейка не создала Python-окружение.")

runner_source = r"""
from pathlib import Path
import nbformat
from nbclient import NotebookClient

app_dir = Path("/content/Applio-vocoders")
report = Path("/content/applio-vocoders-smoke.executed.ipynb")
kernel_name = "applio-vocoders"

cells = [
    nbformat.v4.new_code_cell(
        "import sys, torch\n"
        "print('python:', sys.version)\n"
        "print('torch:', torch.__version__)\n"
        "print('cuda build:', torch.version.cuda)\n"
        "print('cuda available:', torch.cuda.is_available())\n"
    ),
    nbformat.v4.new_code_cell(
        "import compileall\n"
        "from pathlib import Path\n"
        "targets = ['app.py', 'core.py', 'rvc', 'tabs']\n"
        "for target in targets:\n"
        "    ok = compileall.compile_dir(target, quiet=1) if Path(target).is_dir() else compileall.compile_file(target, quiet=1)\n"
        "    if not ok:\n"
        "        raise RuntimeError(f'compileall failed: {target}')\n"
        "print('compileall: OK')\n",
        metadata={"tags": ["compile-check"]},
    ),
    nbformat.v4.new_code_cell(
        "import importlib\n"
        "modules = [\n"
        " 'rvc.configs.config',\n"
        " 'rvc.lib.algorithm.generators.bigvgan',\n"
        " 'rvc.lib.algorithm.generators.ddsp',\n"
        " 'rvc.lib.algorithm.generators.ddsp_v2',\n"
        " 'rvc.lib.algorithm.generators.ddsp_v3',\n"
        " 'rvc.lib.algorithm.generators.hifigan',\n"
        " 'rvc.lib.algorithm.generators.hifigan_aa',\n"
        " 'rvc.lib.algorithm.generators.hifigan_cam',\n"
        " 'rvc.lib.algorithm.generators.hifigan_nsf',\n"
        " 'rvc.lib.algorithm.generators.hifigan_pqmf',\n"
        " 'rvc.lib.algorithm.generators.hifigan_snake',\n"
        " 'rvc.lib.algorithm.generators.hiftnet',\n"
        " 'rvc.lib.algorithm.generators.hiftnet2',\n"
        " 'rvc.lib.algorithm.generators.ringformer',\n"
        " 'rvc.lib.algorithm.generators.velocity',\n"
        " 'rvc.lib.algorithm.generators.vocos',\n"
        " 'rvc.lib.algorithm.generators.vocos_v2',\n"
        " 'rvc.lib.algorithm.generators.wavehax',\n"
        " 'rvc.lib.algorithm.generators.wavehax1d',\n"
        " 'rvc.lib.algorithm.generators.wavenext',\n"
        " 'rvc.lib.algorithm.discriminators.CoMBD',\n"
        " 'rvc.lib.algorithm.discriminators.SBD',\n"
        " 'rvc.lib.algorithm.discriminators.mbd',\n"
        " 'rvc.lib.algorithm.discriminators.mpd',\n"
        " 'rvc.lib.algorithm.discriminators.mrd',\n"
        " 'rvc.lib.algorithm.discriminators.msd',\n"
        " 'rvc.lib.algorithm.discriminators.univnet',\n"
        "]\n"
        "errors = []\n"
        "for name in modules:\n"
        "    try:\n"
        "        importlib.import_module(name)\n"
        "        print('OK ', name)\n"
        "    except Exception as error:\n"
        "        errors.append((name, repr(error)))\n"
        "        print('ERR', name, repr(error))\n"
        "if errors:\n"
        "    raise RuntimeError('Import failures: ' + repr(errors))\n",
        metadata={"tags": ["vocoder-imports"]},
    ),
    nbformat.v4.new_code_cell(
        "from rvc.configs.config import Config\n"
        "config = Config()\n"
        "print('device:', config.device)\n"
        "print('precision:', config.get_precision())\n",
        metadata={"tags": ["runtime-config"]},
    ),
]

notebook = nbformat.v4.new_notebook(
    cells=cells,
    metadata={
        "kernelspec": {
            "display_name": "Applio Vocoders Test",
            "language": "python",
            "name": kernel_name,
        }
    },
)

client = NotebookClient(
    notebook,
    timeout=900,
    kernel_name=kernel_name,
    resources={"metadata": {"path": str(app_dir)}},
    allow_errors=False,
)
client.execute()
nbformat.write(notebook, report)

for index, cell in enumerate(notebook.cells, start=1):
    for output in cell.get("outputs", []):
        text = output.get("text")
        if text:
            print(f"[cell {index}] {text}", end="")
print(f"\nExecuted notebook report: {report}")
"""

RUNNER.write_text(textwrap.dedent(runner_source), encoding="utf-8")
print("Тест стартовал отдельным процессом. Лог будет идти сюда.\n")

process = subprocess.Popen(
    [str(PYTHON), str(RUNNER)],
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"Фоновый тест завершился с кодом {return_code}")

if not REPORT.exists():
    raise RuntimeError("nbclient не создал отчёт выполненного ноутбука.")

print("✅ Все тестовые ячейки выполнены. Отчёт:", REPORT)


In [ ]:
# @title 3. Синхронизация с Google Drive (необязательно)
sync_mode = "не синхронизировать"  # @param ["не синхронизировать", "Drive → Colab", "Colab → Drive"]
model_name = ""  # @param {type:"string"}

from pathlib import Path
import shutil

if sync_mode == "не синхронизировать":
    print("Синхронизация пропущена.")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    if not model_name.strip():
        raise ValueError("Укажи имя модели.")

    local = Path("/content/Applio-vocoders/logs") / model_name.strip()
    remote = Path("/content/drive/MyDrive/ApplioBackup") / model_name.strip()
    source, destination = (
        (remote, local)
        if sync_mode == "Drive → Colab"
        else (local, remote)
    )

    if not source.exists():
        raise FileNotFoundError(source)

    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print("✅ Синхронизация завершена:", destination)


In [ ]:
# @title 4. Запустить Applio в фоне
share_link = True  # @param {type:"boolean"}
port = 6969  # @param {type:"integer"}
startup_timeout = 600  # @param {type:"integer"}

from pathlib import Path
import os
import re
import signal
import socket
import subprocess
import time

APP_DIR = Path("/content/Applio-vocoders")
PYTHON = Path("/content/applio-vocoders-env/bin/python")
PID_FILE = Path("/content/applio-vocoders.pid")
LOG_FILE = Path("/content/applio-vocoders.log")

if not (APP_DIR / "app.py").exists() or not PYTHON.exists():
    raise RuntimeError("Установка не завершена.")

def process_is_alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def stop_previous_server():
    if not PID_FILE.exists():
        return
    try:
        pid = int(PID_FILE.read_text(encoding="utf-8").strip())
    except ValueError:
        PID_FILE.unlink(missing_ok=True)
        return
    if process_is_alive(pid):
        print("Останавливаю предыдущий сервер:", pid)
        try:
            os.killpg(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        for _ in range(20):
            if not process_is_alive(pid):
                break
            time.sleep(0.5)
        if process_is_alive(pid):
            try:
                os.killpg(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
    PID_FILE.unlink(missing_ok=True)

def port_is_open(host, target_port):
    with socket.socket() as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, target_port)) == 0

stop_previous_server()
LOG_FILE.write_text("", encoding="utf-8")

requested_port = port
for candidate in range(requested_port, max(1024, requested_port - 10), -1):
    if not port_is_open("127.0.0.1", candidate):
        port = candidate
        break
else:
    raise RuntimeError("Не найден свободный порт рядом с указанным.")

if port != requested_port:
    print(f"Порт {requested_port} занят, использую {port}.")

command = [str(PYTHON), "app.py", "--port", str(port)]
if share_link:
    command.append("--share")

environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"
environment["TF_CPP_MIN_LOG_LEVEL"] = "3"

log_handle = LOG_FILE.open("a", encoding="utf-8")
process = subprocess.Popen(
    command,
    cwd=APP_DIR,
    env=environment,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
PID_FILE.write_text(str(process.pid), encoding="utf-8")
print("Сервер запущен в фоне. PID:", process.pid)
print("Лог:", LOG_FILE)

deadline = time.time() + max(30, startup_timeout)
last_size = 0
ready = False
while time.time() < deadline:
    if process.poll() is not None:
        log_handle.close()
        tail = LOG_FILE.read_text(encoding="utf-8", errors="replace")[-8000:]
        raise RuntimeError(
            f"Applio завершился с кодом {process.returncode}.\nПоследний лог:\n{tail}"
        )

    if port_is_open("127.0.0.1", port):
        ready = True
        break

    size = LOG_FILE.stat().st_size
    if size != last_size:
        last_size = size
        print("Инициализация...", size, "байт лога")
    time.sleep(2)

log_handle.flush()
log_handle.close()
if not ready:
    tail = LOG_FILE.read_text(encoding="utf-8", errors="replace")[-8000:]
    raise TimeoutError(
        f"Сервер не открыл порт {port} за {startup_timeout} секунд.\n"
        f"Последний лог:\n{tail}"
    )

print(f"✅ Локальный адрес: http://127.0.0.1:{port}")

if share_link:
    share_deadline = time.time() + 90
    share_url = None
    pattern = re.compile(r"https://[a-zA-Z0-9.-]+\.gradio\.live")
    while time.time() < share_deadline and process.poll() is None:
        log_text = LOG_FILE.read_text(encoding="utf-8", errors="replace")
        match = pattern.search(log_text)
        if match:
            share_url = match.group(0)
            break
        time.sleep(2)

    if share_url:
        print("✅ Публичная ссылка:", share_url)
    else:
        print("⚠️ Сервер работает, но публичная ссылка ещё не появилась.")
        print("Проверь последние строки:", LOG_FILE)

print("Ячейка завершена, сервер продолжает работать в фоне.")
